**Matrix algebra.**
A **matrix** is a 2-D rectangular array of numbers.
In NumPy a matrix is a 2-D `ndarray` with shape `(rows, cols)`.
This notebook covers four fundamental matrix operations:

- **Multiplication** $\mathbf{A}\cdot\mathbf{B}$: only valid when the number of
  columns in $\mathbf{A}$ equals the number of rows in $\mathbf{B}$
- **Determinant** $|\mathbf{A}|$: a scalar that encodes whether the matrix is
  invertible ($|\mathbf{A}| \neq 0$) and how it scales area/volume
- **Cramer's rule**: solving a linear system $\mathbf{A}\mathbf{x}=\mathbf{b}$
  by expressing each unknown as a ratio of determinants
- **Direct solve**: `np.linalg.solve` as the numerically preferred alternative

In [ ]:
"""matrix_algebra.ipynb"""

# Cell 01 - Create matrices A (2x3) and B (3x2)

import numpy as np
from IPython.display import Math, display

from qis101_utils import as_latex


def print_ndarray_info(name, a):
    print(f"Type of {name} is {type(a).__name__}")
    print(f"Number of dimensions of {name} = {a.ndim}")
    print(f"Shape of dimensions of {name} = {a.shape}")
    print(f"Length of {name} = {len(a)}")
    print(f"Size of {name} = {a.size}")


a = np.array([[4, 5, 8], [1, 9, 7]])  # shape (2, 3)
b = np.array([[2, 4], [6, 1], [5, 9]])  # shape (3, 2)

print_ndarray_info("a", a)
display(as_latex(a, prefix=r"\mathbf{A}="))

print_ndarray_info("b", b)
display(as_latex(b, prefix=r"\mathbf{B}="))


---
**Matrix multiplication - shape rules.**
The product $\mathbf{A}\cdot\mathbf{B}$ is valid only when
the inner dimensions match: an $(m \times k)$ matrix times a $(k \times n)$
matrix gives an $(m \times n)$ result.
Importantly, matrix multiplication is **not commutative** in general:
$\mathbf{A}\cdot\mathbf{B} \neq \mathbf{B}\cdot\mathbf{A}$.
Here $\mathbf{A}$ is $(2\times 3)$ and $\mathbf{B}$ is $(3\times 2)$,
so $\mathbf{A}\cdot\mathbf{B}$ gives $(2\times 2)$
and $\mathbf{B}\cdot\mathbf{A}$ gives $(3\times 3)$ - different shapes entirely.

In [ ]:
# Cell 02 - A * B: (2x3) @ (3x2) -> (2x2)

c = np.matmul(a, b)
display(as_latex(c, prefix=r"\mathbf{A\cdot B}="))

In [ ]:
# Cell 03 - B * A: (3x2) @ (2x3) -> (3x3)

c = np.matmul(b, a)
display(as_latex(c, prefix=r"\mathbf{B\cdot A}="))

---
**Dimension mismatch raises `ValueError`.**
If $\mathbf{A}$ is extended to $(3\times 3)$ by appending a zero row,
then $\mathbf{B}\cdot\mathbf{A}$ requires a $(3\times 2)$ times a $(3\times 3)$ product.
The inner dimensions are 2 and 3 - they do not match - so NumPy raises a `ValueError`.
The `try/except` block catches this and prints the error message
rather than halting the notebook.

In [ ]:
# Cell 04 - Extending A to (3x3) makes B*A invalid: (3x2) @ (3x3) -> ValueError

a = np.array([[4, 5, 8], [1, 9, 7], [0, 0, 0]])

display(as_latex(a, prefix=r"\mathbf{A}="))
display(as_latex(b, prefix=r"\mathbf{B}="))

try:
    c = np.matmul(b, a)  # (3x2) @ (3x3): inner dims 2 != 3 -> ValueError
    display(as_latex(c, prefix=r"\mathbf{C}="))
except ValueError as e:
    print("ValueError during matrix multiplication:", e)

---
**The determinant.**
The determinant $|\mathbf{A}|$ is defined only for square matrices.
For a $2\times 2$ matrix it is $|\mathbf{A}| = ad - bc$.
The raw floating-point result may carry a small rounding error;
`np.round` is used to display the clean exact value alongside the raw result.

In [ ]:
# Cell 05 - Determinant of a 2x2 matrix
# Exact value: 8*2 - 3*4 = 4

a = np.array([[8, 3], [4, 2]])
display(as_latex(a, prefix=r"\mathbf{A}="))

display(Math(rf"|A|={np.linalg.det(a)}"))  # shows floating-point noise
display(Math(rf"|A|={np.round(np.linalg.det(a), 10)}"))  # rounded to 10 decimal places

---
**Determinant of a large matrix.**
NumPy computes determinants via LU decomposition, which scales efficiently
to large matrices.
The determinant of a $10\times 10$ integer matrix can be a very large integer;
the `:,.0f` format string adds thousands separators for readability.

In [ ]:
# Cell 06 - Determinant of a 10x10 matrix

a = np.array(
    [
        [3, 4, 9, 8, -5, -2, 2, 7, 7, 4],
        [6, -10, -10, -10, -6, -10, 1, 0, -4, -5],
        [-4, -4, -4, 8, -5, 9, 4, 0, -4, 9],
        [-4, 6, -5, 8, 0, 3, 1, -4, -6, -6],
        [-6, -9, 9, 9, -2, 9, -5, -2, -2, -3],
        [-8, 0, 0, 0, 10, -3, 5, 0, -4, 9],
        [8, 4, 8, -9, 4, 8, -6, 1, 9, 2],
        [2, 1, -1, 4, -6, -10, 1, -6, 6, 7],
        [-6, -5, 4, -6, -5, -6, -10, -1, -2, 7],
        [1, 10, -10, -4, -8, 7, 5, -1, 6, -6],
    ]
)

display(as_latex(a, prefix=r"\mathbf{A}="))
display(Math(rf"|A|={np.linalg.det(a):,}"))  # with floating-point noise
display(Math(rf"|A|={np.linalg.det(a):,.0f}"))  # rounded to nearest integer

---
**Cramer's rule for solving linear systems.**
Given $\mathbf{A}\mathbf{x} = \mathbf{b}$, Cramer's rule expresses each unknown
as a ratio of determinants.
For unknown $x_i$, replace column $i$ of $\mathbf{A}$ with $\mathbf{b}$
to form matrix $\mathbf{A}_i$, then:

$$x_i = \frac{|\mathbf{A}_i|}{|\mathbf{A}|}$$

This requires $|\mathbf{A}| \neq 0$ (the system must be non-singular).
Cramer's rule is elegant but computationally expensive for large systems;
`np.linalg.solve` uses a faster LU-based approach and is preferred in practice.

In [ ]:
# Cell 07 - Coefficient matrix and right-hand side vector for a 3x3 linear system

coeffs = np.array([[4, 5, -2], [7, -1, 2], [3, 1, 4]])
vals = np.array([-14, 42, 28])

display(as_latex(coeffs, prefix=r"\mathbf{Coeffs}="))
display(as_latex(vals, column=True, prefix=r"\mathbf{Vals}="))

In [ ]:
# Cell 08 - Solve using Cramer's rule

# For each unknown, replace that column of coeffs with vals
xa = np.copy(coeffs)
xa[:, 0] = vals  # replace column 0 for x

ya = np.copy(coeffs)
ya[:, 1] = vals  # replace column 1 for y

za = np.copy(coeffs)
za[:, 2] = vals  # replace column 2 for z

det_coeffs = np.linalg.det(coeffs)  # det(A) must be nonzero

x = np.linalg.det(xa) / det_coeffs
y = np.linalg.det(ya) / det_coeffs
z = np.linalg.det(za) / det_coeffs

display(Math(rf"x={x:,.0f}"))
display(Math(rf"y={y:,.0f}"))
display(Math(rf"z={z:,.0f}"))

---
**Direct solve via LU decomposition.**
`np.linalg.solve` finds the solution to $\mathbf{A}\mathbf{x}=\mathbf{b}$
using LU decomposition internally.
It is faster and numerically more stable than Cramer's rule for large systems
and produces the same result here.

In [ ]:
# Cell 09 - Solve the same system using np.linalg.solve (LU decomposition)

sol = np.linalg.solve(coeffs, vals)
display(as_latex(sol, column=True, prefix=r"\mathbf{x}="))